<a href="https://colab.research.google.com/github/kinjaljoshi/mech-interp-from-scratch/blob/master/linear_probes_gpt2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1. Load Dataset

In [1]:
import torch, torch.nn as nn
from datasets import load_dataset

torch.manual_seed(0)
ds = load_dataset("nyu-mll/glue", "sst2", split="train")

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

sst2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

sst2/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  148kB            

sst2/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [2]:
ds = ds.filter(lambda x: 4 <=len(x['sentence'].split()) <= 15)
ds = ds.shuffle(seed=0).select(range(1200))

texts = [x.strip() for x in ds['sentence']]
y = torch.tensor(ds['label']).float()

print("examples:", len(texts), "| positive rate:", y.mean().item())
print(texts[:3])

Filter:   0%|          | 0/67349 [00:00<?, ? examples/s]

examples: 1200 | positive rate: 0.5525000095367432
["ca n't help but be drawn in by the sympathetic characters", 'hugely enjoyable in its own right though not really faithful to', 'now , told by hollywood , and much more ordinary for it']


In [3]:
!pip install transformer_lens

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 54.6 MB/s  0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12526 sha256=84e9ddc1ef9b57ee8c76eaba1cbb81658ee83ad3b8385f3bfb034050f4c932fe
  Stored in directory: /root/.cache/pip/wheels/f6/6e/1a/710143d0e50e827ae5b60a47a6d43d715cf1c6e35881a05530
Successfully built transformers-stream-generator
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [transformer_lens]


#2. Initialize model using Transformer Lens

In [4]:
from transformer_lens import TransformerBridge


model = TransformerBridge.boot_transformers("gpt2")

TransformerBridge.enable_compatibility_mode(model)
n_layers = model.cfg.n_layers
d = model.cfg.d_model

# resid_post = the residual stream right after each block finishes.
# One hook name per layer, so we can read what the model "knows" at each depth.
feats = torch.zeros(n_layers, len(texts), d)
feats.shape

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

torch.Size([12, 1200, 768])

#3. Capturing activation using Residual Stream Hooks on all layers

In [7]:
# run_with_cache runs the model on one snippet and saves the activations for
# names_filter keeps only our 12 hooks, so we don't store everything.
hook =[f"blocks.{l}.hook_resid_post" for l in range(n_layers)]

with torch.no_grad():
  for i, t in enumerate(texts):
    _, cache = model.run_with_cache(t, names_filter=lambda x: x in hook)
    #print('Cache Shape with keys - ', cache.keys())
    for l, h in enumerate(hook):
      feats[l, i] = cache[h][0, -1].cpu()
    if i % 200 == 0:
      print("done", i + 200, "of", len(texts))


done 200 of 1200
done 400 of 1200
done 600 of 1200
done 800 of 1200
done 1000 of 1200
done 1200 of 1200


In [8]:
print(feats.shape)  # expect torch.Size([12, 1200, 768])
print("any NaNs:", torch.isnan(feats).any().item())  # sanity check: should be False
print("per-layer mean norm:", [round(feats[l].norm(dim=1).mean().item(), 1) for l in range(n_layers)])

torch.Size([12, 1200, 768])
any NaNs: False
per-layer mean norm: [58.3, 58.5, 63.1, 68.9, 75.4, 82.1, 93.0, 109.5, 129.3, 155.6, 238.3, 466.1]


1. The extraction worked: the shape is [12, 1200, 768], with no NaNs. The norms confirm what the “residual stream is built by adding”.
2. The average norm grows from 58 at layer 0 to 466 at layer 11, about 8×, and the growth accelerates in the last few layers. Late blocks write much larger updates into the stream than early ones.

This matters for the probe in two ways:

1. Scale differs by layer. The layer 11 vectors are about 8× longer than layer 0’s. This is why the probe standardizes each feature before fitting: without it, learning would behave very differently from layer to layer, and the comparison between layers wouldn’t be fair.
2. A few dimensions probably dominate the norm. GPT-2’s residual stream is known for a handful of dimensions with outsized values.

#4. Training Linear probe on captured activtions and measuring accuracy

In [16]:
tr, te = slice(0, 900), slice(900, 1200)

In [25]:
#divide feats into training and test samples and train the probe and then measure accurary
#feats.shape -> (12, 1200, 768)
probes = {}
def train_probe(mode, l, X, y, steps =300):
    # X: (1200, 768) activations from ONE layer. y: the 0/1 labels.
    # Standardize each of the 768 dims using TRAIN rows only, so no test information
    # leaks into training. The 1e-6 avoids dividing by zero for a constant dimension.
  probe = nn.Linear(768, 1)
  mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-6
  Xn = X - mu / sd
    # weight_decay gently penalizes large weights, which stops the probe from
    # memorizing the 900 training examples
  opt = torch.optim.Adam(probe.parameters(), lr = 1e-2, weight_decay=1e-2)
  for i in range(steps):
    # logistic-regression loss: pushes the score above 0 for positive, below 0 for negative
    loss = nn.functional.binary_cross_entropy_with_logits(probe(Xn[tr]).squeeze(-1), y[tr])
    opt.zero_grad()
    loss.backward()
    opt.step()
  probename = f"probe{l}"
  probes[probename] = probe
  #check accuracy
  with torch.no_grad():
    probe.eval()
    print(f"{mode} - {l:5d} | {((probe(Xn[te]).squeeze(-1) >0).float() == y[te]).float().mean()}")

for l in range(n_layers):
  train_probe("Real Labels", l, feats[l], y)

torch.manual_seed(1)
y_shuff = y[torch.randperm(len(y))]
for l in range(n_layers):
  train_probe("Shuffeled Labels", l, feats[l], y_shuff)



Real Labels -     0 | 0.6566666960716248
Real Labels -     1 | 0.6966666579246521
Real Labels -     2 | 0.7099999785423279
Real Labels -     3 | 0.7400000095367432
Real Labels -     4 | 0.7733333110809326
Real Labels -     5 | 0.7400000095367432
Real Labels -     6 | 0.7599999904632568
Real Labels -     7 | 0.7666666507720947
Real Labels -     8 | 0.7733333110809326
Real Labels -     9 | 0.7433333396911621
Real Labels -    10 | 0.7866666913032532
Real Labels -    11 | 0.7733333110809326
Shuffeled Labels -     0 | 0.4699999988079071
Shuffeled Labels -     1 | 0.46666666865348816
Shuffeled Labels -     2 | 0.49666666984558105
Shuffeled Labels -     3 | 0.4866666793823242
Shuffeled Labels -     4 | 0.5099999904632568
Shuffeled Labels -     5 | 0.4866666793823242
Shuffeled Labels -     6 | 0.4933333396911621
Shuffeled Labels -     7 | 0.49666666984558105
Shuffeled Labels -     8 | 0.5099999904632568
Shuffeled Labels -     9 | 0.49666666984558105
Shuffeled Labels -    10 | 0.579999983310699

The control now works: the shuffled labels score about 50% at most layers, while the real labels score 66-79%. Here’s what the numbers show.

What the real column says

1. Layer 0 already gets 66%. The last token carries some sentiment on its own. This is the floor for “how much a single token tells you”, so the other layers should be judged against 66%, not against 50%.
3. Accuracy climbs from 66% to about 77% by layer 4, a gain of about 11 points. This is where sentiment becomes more readable as the model mixes context into the last token.
4. After layer 4 it plateaus between 74% and 79%. The standard error on 300 test snippets is about 2.4 points at 77%, so the wiggles between layers 4 and 11 (for example 74.3% at layer 9 vs 78.7% at layer 10) are within noise.